# Agent 2 — Step C: Train XGBoost + Evaluate + Inference Test  (v2 — no grade leak)

**Input:** `train.csv`, `test.csv`, `encoders.pkl`, `features.txt`

**Output:** `price_model.pkl` (trained XGBoost) + honest evaluation metrics + a working `predict_base_price()` function.

**What changed from v1:**
- ❌ Dropped `grade_enc` from features. Model now predicts the BASE price for (millet, state, district, year, month). Grade adjustment is applied in `price_agent.py` (`{A: 1.10, B: 1.00, C: 0.88}`).
- 📈 Expect higher MAPE (~12–18%) than v1 (~5%). The previous low MAPE was largely from learning the deterministic grade multiplier — that was leakage, not real predictive skill.

**Why XGBoost (not LSTM):**
- ~1,853 monthly rows split across millet × state × district means each individual time series is only ~50–60 points → LSTMs overfit on sequences this short.
- XGBoost handles categorical + temporal features well and trains in seconds.
- Target accuracy: MAPE < 20% (a realistic target for this dataset size).

## Cell 1 — Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np
import pickle

DRIVE = "/content/drive/MyDrive/MilletSaarthi"

train = pd.read_csv(f"{DRIVE}/train.csv")
test  = pd.read_csv(f"{DRIVE}/test.csv")
with open(f"{DRIVE}/features.txt") as f:
    FEATURES = f.read().strip().split(",")
with open(f"{DRIVE}/encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

TARGET = "modal_price"
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {FEATURES}")
assert "grade_enc" not in FEATURES, "grade_enc should not be in features — re-run Step B"

## Cell 2 — Train XGBoost

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric="mae"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print("\n✅ Training done")

## Cell 3 — Evaluate on test set (honest metrics)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
r2   = r2_score(y_test, preds)

print(f"MAE  : ₹{mae:.2f} per quintal")
print(f"RMSE : ₹{rmse:.2f} per quintal")
print(f"MAPE : {mape:.2f}%   (target: <20%)")
print(f"R²   : {r2:.4f}")

# Feature importance
print("\nFeature importance:")
imp = pd.DataFrame({"feature": FEATURES, "importance": model.feature_importances_})
print(imp.sort_values("importance", ascending=False).to_string(index=False))

## Cell 4 — Save model

In [ ]:
import os
os.makedirs(f"{DRIVE}/models", exist_ok=True)

with open(f"{DRIVE}/models/price_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Saved models/price_model.pkl")
print("✅ encoders.pkl already saved from Step B (no grade encoder)")

## Cell 5 — Inference test (the function Agent 2 will use)

Note: this returns the BASE price (no grade adjustment). The grade multiplier is applied in `price_agent.py` after Agent 1 predicts the grade.

In [ ]:
def predict_base_price(millet: str, state: str, district: str,
                       year: int, month: int) -> float:
    """
    Predict the base modal price per quintal (no grade adjustment).

    Args:
        millet: 'jowar' | 'bajra' | 'ragi'
        state:  e.g. 'Maharashtra'
        district: e.g. 'Pune'
        year, month: prediction month

    Returns: predicted base price (₹/quintal). Grade adjustment is applied
    downstream in price_agent.py.
    """
    def season_of(m):
        if m in (6, 7, 8, 9): return "kharif"
        if m in (10, 11, 12, 1, 2, 3): return "rabi"
        return "summer"

    row = {
        "millet_enc":   encoders["millet"].transform([millet])[0],
        "state_enc":    encoders["state"].transform([state])[0],
        "district_enc": encoders["district"].transform([district])[0],
        "season_enc":   encoders["season"].transform([season_of(month)])[0],
        "year":         year,
        "month":        month,
        "month_sin":    np.sin(2 * np.pi * month / 12),
        "month_cos":    np.cos(2 * np.pi * month / 12),
    }
    X = pd.DataFrame([row])[FEATURES]
    return float(model.predict(X)[0])


# Test 1 — Jowar in Pune, May 2026
base = predict_base_price("jowar", "Maharashtra", "Pune", 2026, 5)
print(f"Jowar / Pune / 2026-05  base price: ₹{base:.2f}/q")
print(f"  Grade A would be: ₹{base*1.10:.2f}")
print(f"  Grade B would be: ₹{base*1.00:.2f}")
print(f"  Grade C would be: ₹{base*0.88:.2f}")

# Test 2 — Bajra in Ahmednagar
base = predict_base_price("bajra", "Maharashtra", "Ahmednagar", 2026, 5)
print(f"\nBajra / Ahmednagar / 2026-05  base price: ₹{base:.2f}/q")

# Test 3 — Ragi in Bangalore
base = predict_base_price("ragi", "Karnataka", "Bangalore", 2026, 6)
print(f"Ragi / Bangalore / 2026-06  base price: ₹{base:.2f}/q")